<a href="https://colab.research.google.com/github/huzaifa003/skin_disease_identification/blob/master/GPT_Questions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get install poppler-utils

In [ ]:
!pip install -Uqqq pip --progress-bar off
!pip install -qqq langchain==0.0.173 --progress-bar off
!pip install -qqq chromadb==0.3.23 --progress-bar off
!pip install -qqq pypdf==3.8.1 --progress-bar off
!pip install -qqq pygpt4all==1.1.0 --progress-bar off
!pip install -qqq pdf2image==1.16.3 --progress-bar off

In [ ]:
!gdown 1DpFisoGXsQbpQJvijuvxkLW_pg-FUUMF

In [ ]:
!wget https://gpt4all.io/models/ggml-gpt4all-j-v1.3-groovy.bin

In [ ]:
from langchain.chains import RetrievalQA
from langchain.document_loaders import PyPDFLoader
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.llms import GPT4All
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from pdf2image import convert_from_path

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
images = convert_from_path("Derm.pdf", dpi=88)
len(images)

In [ ]:
images[0]

In [ ]:
images[1]

## Load Data

In [ ]:
loader = PyPDFLoader("Derm.pdf")

In [ ]:
documents = loader.load_and_split()

In [ ]:
len(documents)

In [ ]:
print(documents[5].page_content)

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=64)
texts = text_splitter.split_documents(documents)

In [ ]:
len(texts)

In [ ]:
print(texts[20].page_content)

## Create Embeddings

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
db = Chroma.from_documents(texts, embeddings, persist_directory="db")

## Create Chain

In [ ]:
model_n_ctx = 1000
model_path = "./ggml-gpt4all-j-v1.3-groovy.bin"
llm = GPT4All(model=model_path, n_ctx=1000, backend="gptj", verbose=False)

In [ ]:
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=db.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True,
    verbose=False,
)

## Ask Questions

In [ ]:
%%time
res = qa(
    "What are dermatology problems? Extract from text"
)

In [ ]:
res

In [ ]:
print(res["result"])

In [ ]:
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=db.as_retriever(search_kwargs={"k": 1}),
    return_source_documents=True,
    verbose=False,
)

In [ ]:
%%time
res = qa(
    "Cause of Staphylococcal scalded skin syndrome? Extract in text"
)

In [ ]:
print(res["result"])

## References

- [GPT4All](https://gpt4all.io/)
- [MICROSOFT 2022 ANNUAL REPORT](https://www.microsoft.com/investor/reports/ar22/download-center/)